# Rimworld-Statistics：コミット間性能比較

**Colabブラウザ版・CPUランタイム**で2つのGitコミットを同一VM内で比較します。既存の `SaveImportBenchmark`（コールド・ウォーム取り込み）を交互に繰り返して実行し、中央値と相対比を出します。オプションで `ProfileBenchmark` も比較できます。

- 対象は**mainの履歴内に存在するコミット**です。初期値は現行 `HEAD` と1コミット前の `HEAD~1`。
- 合成データのみを使用し、RimWorld本体・実セーブは不要です。
- Colabランタイム間の比較は不安定なので、同じランタイム内でA/Bを交互に実行します。これだけでゲーム内TPS改善を証明するものではありません。

## 1. 比較条件

`BASE_REF` と `TARGET_REF` に、main履歴内のコミットSHAや `HEAD~2` などを指定できます。実行回数を増やすほどブレを確認しやすくなりますが、計算資源を消費します。

In [ ]:
BASE_REF = "HEAD~1"  # @param {type:"string"}
TARGET_REF = "HEAD"  # @param {type:"string"}
REPEATS = 3  # @param {type:"slider",min:2,max:10,step:1}
SAVE_COUNTS = (55, 200)
PADDING_BYTES = 262144
RUN_PROFILE_BENCHMARK = True  # @param {type:"boolean"}
assert 2 <= int(REPEATS) <= 10
assert all(1 <= count <= 3000 for count in SAVE_COUNTS)

## 2. 準備：Git履歴・.NET 10・2つの作業ツリー

既存ファイルは編集しません。比較用の2つのソースツリーをColabの専用一時フォルダへ作成します。

In [ ]:
import csv, io, json, os, re, shutil, subprocess, urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path("/content/rws-colab-ab")
REPOSITORY = BASE / "repository"
OUT = Path("/content/rws-colab-output")
OUT.mkdir(parents=True, exist_ok=True)

def command(args, cwd=None, timeout=1800):
    result = subprocess.run([str(x) for x in args], cwd=cwd,
                            env=os.environ.copy(), capture_output=True,
                            text=True, errors="replace", timeout=timeout)
    if result.returncode:
        raise RuntimeError(f"コマンド失敗 ({result.returncode}): {result.stderr[-1400:]}")
    return result.stdout.strip()

def prepare():
    if not (REPOSITORY / ".git").is_dir():
        command(["git", "clone", "--depth", "100",
                 "https://github.com/pengin0503/Rimworld-Statistics.git", REPOSITORY],
                timeout=300)
    else:
        command(["git", "-C", REPOSITORY, "fetch", "--depth", "100", "origin", "main"],
                timeout=300)
        command(["git", "-C", REPOSITORY, "checkout", "--detach", "FETCH_HEAD"])
    dotnet_root = Path("/content/rws-dotnet")
    if not (dotnet_root / "dotnet").is_file():
        installer = Path("/content/rws-dotnet-install.sh")
        urllib.request.urlretrieve("https://dot.net/v1/dotnet-install.sh", installer)
        command(["bash", installer, "--channel", "10.0", "--install-dir", dotnet_root],
                timeout=900)
    os.environ["DOTNET_ROOT"] = str(dotnet_root)
    os.environ["PATH"] = str(dotnet_root) + os.pathsep + os.environ["PATH"]
    os.environ["DOTNET_CLI_TELEMETRY_OPTOUT"] = "1"
    os.environ["DOTNET_SKIP_FIRST_TIME_EXPERIENCE"] = "1"
    version = command(["dotnet", "--version"])
    assert version.startswith("10."), f".NET 10が必要です: {version}"
    print("SDK:", version)

prepare()
for ref in (BASE_REF, TARGET_REF):
    if not re.fullmatch(r"[A-Za-z0-9_./~^+-]{1,100}", ref):
        raise ValueError("Git refに使用できない文字があります。")

def resolve(ref):
    try:
        return command(["git", "-C", REPOSITORY, "rev-parse",
                        "--verify", ref + "^{commit}"])
    except RuntimeError as exc:
        raise ValueError("コミットが見つかりません。main履歴内のSHAを指定してください。") from exc

sha_by_label = {"base": resolve(BASE_REF), "target": resolve(TARGET_REF)}
if sha_by_label["base"] == sha_by_label["target"]:
    raise ValueError("BASE_REFとTARGET_REFが同じコミットです。")
source_by_label = {}
for label, sha in sha_by_label.items():
    path = BASE / label
    if path.exists():
        command(["git", "-C", REPOSITORY, "worktree", "remove", "--force", path])
    command(["git", "-C", REPOSITORY, "worktree", "add", "--detach", path, sha])
    source_by_label[label] = path
print("base:", sha_by_label["base"][:12], "/ target:", sha_by_label["target"][:12])

## 3. コンパイルして、A/Bを交互に測定

各ツリーを一度ビルドしてから `--no-build` で測定します。毎回の実行が自身の一時ディレクトリに合成セーブを作り、コールド／ウォーム動作を確認します。

In [ ]:
for label, path in source_by_label.items():
    command(["dotnet", "build",
             path / "tools/SaveImportBenchmark/SaveImportBenchmark.csproj",
             "-c", "Release", "-p:UseNuGetRefs=true"], cwd=path)
    if RUN_PROFILE_BENCHMARK:
        command(["dotnet", "build",
                 path / "tools/ProfileBenchmark/ProfileBenchmark.csproj",
                 "-c", "Release", "-p:UseNuGetRefs=true"], cwd=path)
    print(label, "ビルド成功")

records = []
for count in SAVE_COUNTS:
    for repeat in range(1, REPEATS + 1):
        labels = ("base", "target") if repeat % 2 else ("target", "base")
        for label in labels:
            path = source_by_label[label]
            output = command(
                ["dotnet", "run", "-c", "Release",
                 "--project", path / "tools/SaveImportBenchmark/SaveImportBenchmark.csproj",
                 "--no-build", "--no-launch-profile", "--", count, PADDING_BYTES],
                cwd=path)
            rows = list(csv.DictReader(io.StringIO(output)))
            if len(rows) != 1:
                raise AssertionError("SaveImportBenchmarkのCSV行数が不正です。")
            item = {k: float(v) if "." in v else int(v) for k, v in rows[0].items()}
            if item["saves"] != count or item["home_partial"] != count:
                raise AssertionError("ベンチマークの正常性チェックに失敗しました。")
            item.update(label=label, commit=sha_by_label[label], repeat=repeat)
            records.append(item)
    print(f"{count}セーブ: {REPEATS}反復 × 2コミット完了")

bench = pd.DataFrame.from_records(records)
bench.to_csv(OUT / "commit-import-comparison.csv", index=False)
display(bench[["label", "repeat", "saves", "cold_ms", "warm_ms",
               "cold_alloc_bytes", "warm_alloc_bytes"]])

## 4. 中央値とグラフ

`base/target` は1より大きいとtargetが速いことを意味します。各反復のバラツキを表で確認してください。スピード差に有意性の保証はありません。

In [ ]:
summary = (bench.groupby(["label", "saves"], as_index=False)
           .agg(cold_median_ms=("cold_ms", "median"),
                warm_median_ms=("warm_ms", "median"),
                cold_alloc_median=("cold_alloc_bytes", "median"),
                cold_min_ms=("cold_ms", "min"),
                cold_max_ms=("cold_ms", "max")))
pivot = summary.pivot(index="saves", columns="label", values="cold_median_ms")
pivot["base_over_target"] = pivot["base"] / pivot["target"].replace(0, float("nan"))
display(pivot.round(3))
summary.to_csv(OUT / "commit-import-summary.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4))
for label, values in summary.groupby("label"):
    ax.plot(values["saves"], values["cold_median_ms"], marker="o", label=label)
ax.set(xlabel="Synthetic saves", ylabel="Median cold import (ms)",
       title="Same Colab VM: commit A/B import")
ax.grid(alpha=.25)
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "commit-import-comparison.png", dpi=150)
plt.show()
plt.close(fig)

## 5. プロファイル集計の比較（任意）

`RUN_PROFILE_BENCHMARK=True` の場合は、プロフィール直列化・表示用集計・ファイル書き込みなどを合成条件で比較します。こちらは1回ずつの探索的測定です。

In [ ]:
profile_summary = None
if RUN_PROFILE_BENCHMARK:
    data = []
    for label, path in source_by_label.items():
        output = command(
            ["dotnet", "run", "-c", "Release",
             "--project", path / "tools/ProfileBenchmark/ProfileBenchmark.csproj",
             "--no-build", "--no-launch-profile", "--", 100, 500, 1500],
            cwd=path)
        rows = list(csv.DictReader(io.StringIO(output)))
        if len(rows) != 3 or not all("build_view_ms" in row for row in rows):
            raise RuntimeError("ProfileBenchmarkの出力形式が想定と異なります。")
        for row in rows:
            row.update(label=label, commit=sha_by_label[label])
            data.append(row)
    profile_summary = pd.DataFrame(data)
    for column in profile_summary.columns.difference(["label", "commit"]):
        profile_summary[column] = pd.to_numeric(profile_summary[column])
    profile_summary.to_csv(OUT / "commit-profile-comparison.csv", index=False)
    display(profile_summary[["label", "saves", "serialize_ms", "build_view_ms",
                             "disk_commit_ms"]].round(3))
    fig, ax = plt.subplots(figsize=(8, 4))
    for label, values in profile_summary.groupby("label"):
        ax.plot(values["saves"], values["build_view_ms"], marker="o", label=label)
    ax.set(xlabel="Synthetic saves", ylabel="Build view (ms)",
           title="Profile build-view exploratory comparison")
    ax.grid(alpha=.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "commit-profile-comparison.png", dpi=150)
    plt.show()
    plt.close(fig)
else:
    print("プロフィール集計の比較はスキップしました。")

## 6. レポートを保存

CSVとPNG、コミット識別子を含むJSONを作成します。Colabのファイルパネルからダウンロード可能です。合成ベンチマークの結果であり、実ゲームのTPS・FPSの測定ではありません。

In [ ]:
report = {
    "schema": 1, "baseline": sha_by_label["base"],
    "target": sha_by_label["target"],
    "repeats": REPEATS, "save_counts": list(SAVE_COUNTS),
    "padding_bytes": PADDING_BYTES, "profile_exploratory": RUN_PROFILE_BENCHMARK,
    "cold_import_speedup_base_over_target": {
        str(count): float(value) for count, value in pivot["base_over_target"].items()
    },
    "scope": "Synthetic C# microbenchmarks on one temporary Colab VM; not TPS/FPS"
}
(OUT / "commit-performance-report.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("生成したレポート:", sorted(p.name for p in OUT.glob("commit-*")))